# 双边异质性模型：哪些部分真的能被 neural equilibrium layer 加速？

## TL;DR

这个 notebook 做两个可证伪的 synthetic experiments，而不是直接宣称 price-operator 方法更快。

1. **Population representation**：复现当前 KS 风格的 full-distribution-per-agent MLP，并与“两个分布各编码一次、共同状态 broadcast 给 agents”的 shared encoder 比较。$N=800$ 时，两次完整运行测得 forward 约快 $23$--$26\times$，reverse-mode gradient 约快 $19$--$24\times$。
2. **Amortized market clearing**：在一个 households 异质劳动供给、firms 异质劳动需求的市场中，用 DeepSets price predictor 初始化工资，再做 Newton corrections。三个训练 seeds 的 raw price MAPE 约为 4--5%；三步 correction 后 price MAPE 约为 $3.7\times10^{-5}\%$，在线求解约快 $4.4\times$。

结论必须拆开解释：第一个加速来自 shared distribution encoding，并不是 pricing economics 独有；第二个加速才来自 amortizing expensive two-sided market clearing。两者在完整动态 HA 模型中是否仍成立，尚未被这个 toy experiment 证明。


## 1. Context & Methods

我们检验两个计算命题：

- **H1：共享 aggregate representation 能改变 population-size scaling。** 当前 KS implementation 把完整 empirical distribution 重复送入每个 individual policy evaluation，理论上产生接近 $O(N^2)$ 的工作量。shared encoder 应接近 $O(N)$。
- **H2：learned equilibrium-price warm start 能减少昂贵的双边 best-response evaluations。** 网络不负责最终 market clearing，只预测初值；Newton corrector 把 residual 降到 benchmark tolerance。

### Key Assumptions

- 这是 synthetic bilateral labor market，不是完整 infinite-horizon household/firm DP。
- Household labor supply 随工资上升，firm labor demand 由 Cobb--Douglas FOC 给出并随工资下降，因此 equilibrium wage 唯一。
- Exact baseline 使用 bisection；learned solver 使用 DeepSets predictor 加 1--4 步解析 Newton correction。
- 所有 timing 都排除函数第一次 warm-up，但 Julia/Flux compilation 仍会影响首次 training 的 offline cost。
- CPU timing 会随机器负载变化；经济上重要的是 scaling、root-evaluation count、accuracy 和 break-even，而不是某个毫秒数。


In [1]:
using Pkg, Printf, Statistics
function find_project_root(start_directory=pwd())
    directory = abspath(start_directory)
    while true
        isfile(joinpath(directory, "Project.toml")) && return directory
        parent = dirname(directory)
        parent == directory && error("Could not locate Project.toml")
        directory = parent
    end
end
project_root = find_project_root()
Base.active_project() == joinpath(project_root, "Project.toml") || Pkg.activate(project_root)
include(joinpath(project_root, "src", "BilateralAccelerationExperiment.jl"))
using .BilateralAccelerationExperiment
println("Loaded BilateralAccelerationExperiment on Julia ", VERSION)


Loaded BilateralAccelerationExperiment on Julia 1.12.5


## 2. Experiment A: full distribution per agent versus shared encoder

Naive policies have the form

$$g_i=g(x_i,\mu^H,\mu^F,p),$$

where the two complete empirical distributions are copied into every agent column. The shared architecture computes

$$h_H=E_H(\mu^H),\qquad h_F=E_F(\mu^F),$$

once, then evaluates

$$g_i^H=g_H(x_i,h_H,h_F,p),\qquad g_j^F=g_F(y_j,h_H,h_F,p).$$

Both sides use hidden width 32. The comparison measures complete household-plus-firm forward calls and gradients of a scalar policy loss with respect to all parameters.


In [2]:
scaling_rows = benchmark_population_architectures(
    population_sizes=(50, 100, 200, 400, 800),
    hidden=32,
    repeats=5,
    seed=20260806,
)
BilateralAccelerationExperiment.print_scaling_rows(scaling_rows)


N | forward ms naive/shared | speedup | gradient ms naive/shared | speedup
  50 |     0.081 /    0.047 |    1.73x |     0.287 /    0.249 |    1.15x
 100 |     0.728 /    0.074 |    9.89x |     1.187 /    0.255 |    4.66x
 200 |     1.141 /    0.109 |   10.48x |     4.019 /    0.503 |    7.99x
 400 |     4.105 /    0.389 |   10.56x |     8.823 /    1.252 |    7.05x
 800 |    13.087 /    0.566 |   23.13x |    39.179 /    2.026 |   19.34x


### Result A

The gap grows sharply with population size. At $N=800$, the latest full run gives about $23.1\times$ faster forward evaluation and $19.3\times$ faster reverse-mode gradient evaluation; an earlier run gave $25.5\times$ and $23.9\times$, respectively.

This is relevant to the current KS implementation because it repeats the full finite cross-section in every agent input. But it is not uniquely a price-operator result: a properly designed policy-first DeepSets model obtains the same asymptotic benefit. Therefore this experiment supports **shared population encoding**, not by itself a new equilibrium concept.


## 3. Experiment B: learned price warm start plus exact correction

Households differ in wealth and labor efficiency. Firms differ in capital and productivity. At a candidate wage $w$ we compute aggregate effective labor supply $L_H(w;\mu^H)$ and firm labor demand $L_F(w;\mu^F)$. The exact equilibrium solves

$$\mathcal M(w;\mu^H,\mu^F)=L_H(w;\mu^H)-L_F(w;\mu^F)=0.$$

A DeepSets predictor learns

$$w^{(0)}=\mathcal P_\theta[\mu^H,\mu^F],$$

using exact bisection roots from 192 training distributions with 128 households and 128 firms. Online evaluation uses populations of 5,000 on each side, testing size-invariant generalization. The corrector applies

$$w^{(k+1)}=w^{(k)}-\frac{\mathcal M(w^{(k)})}{\partial_w\mathcal M(w^{(k)})}.$$


In [3]:
training = train_price_predictor(
    population_size=128,
    training_states=192,
    validation_states=64,
    hidden=16,
    epochs=250,
    seed=20260807,
)
@printf("offline data generation = %.3f s\n", training.data_seconds)
@printf("offline training wall time = %.3f s\n", training.training_seconds)
@printf("validation price MAPE = %.3f%%\n", 100training.validation_price_mape)


offline data generation = 0.047 s
offline training wall time = 13.218 s
validation price MAPE = 4.907%


In [4]:
correction_rows = NamedTuple[]
for correction_steps in 1:4
    benchmark = benchmark_price_solver(
        training.model,
        population_size=5_000,
        batch_size=32,
        correction_steps=correction_steps,
        repeats=5,
        seed=20260901,
    )
    push!(correction_rows, benchmark)
    @printf(
        "steps %d: speedup %.2fx, corrected MAPE %.6f%%, max residual %.3e, evals %.1f vs %.1f\n",
        correction_steps,
        benchmark.online_speedup,
        100benchmark.corrected_price_mape,
        benchmark.corrected_max_residual,
        benchmark.corrected_mean_evaluations,
        benchmark.exact_mean_evaluations,
    )
end


steps 1: speedup 6.87x, corrected MAPE 0.611378%, max residual 5.291e-02, evals 2.0 vs 23.2
steps 2: speedup 5.65x, corrected MAPE 0.012425%, max residual 2.155e-03, evals 3.0 vs 23.2
steps 3: speedup 4.76x, corrected MAPE 0.000038%, max residual 3.904e-06, evals 4.0 vs 23.2
steps 4: speedup 3.70x, corrected MAPE 0.000031%, max residual 1.192e-07, evals 5.0 vs 23.2


### Result B

The raw predictor is not accurate enough to be accepted as an equilibrium solution. The useful result is predictor--corrector behavior:

- Bisection uses about 23.2 population residual evaluations per state.
- Three Newton corrections use four evaluations including the final residual check, deliver approximately $3.8\times10^{-5}\%$ price MAPE, and retain about $4.8\times$ online speedup in the representative run.
- Four corrections match the benchmark residual more tightly but reduce the speedup to about $3.7\times$ in the latest run.

The network is therefore useful as an amortized warm start, not as a replacement for market clearing.


## 4. Seed robustness

A separate same-process run repeated the 250-epoch training for three seeds and evaluated all models on the same $N=5{,}000$, 32-state test batch with three corrections.

| Seed | Validation MAPE | Test raw MAPE | Corrected MAPE | Online speedup |
|---:|---:|---:|---:|---:|
| 20260806 | 5.085% | 4.229% | 0.000044% | 4.49x |
| 20260807 | 4.907% | 5.597% | 0.000038% | 4.32x |
| 20260808 | 4.441% | 4.739% | 0.000029% | 4.45x |
| Mean | 4.811% | 4.855% | 0.000037% | 4.42x |

The raw predictor varies across seeds, while Newton correction makes final accuracy and online speed much more stable.


## 5. Offline cost and break-even

Online speed is not free. In a fresh Julia process, data generation plus compilation and training took about 13--26 seconds in the observed runs; after compilation, another 250-epoch training took about 4.5 seconds. With a saving of roughly 0.2--0.25 seconds per 32-state batch, the observed break-even is on the order of tens to roughly one hundred comparable batches.

Therefore this method is attractive for estimation, policy optimization, parameter sweeps, transition-path iteration, or repeated simulations. It is much less compelling when solving one equilibrium at one calibration.


## 6. Limitations

- The synthetic responses are closed-form and monotone; a full dynamic household/firm problem is substantially harder.
- The predictor can succeed partly because this toy equilibrium has low-dimensional sufficient distribution features. A genuinely infinite-dimensional price law may require a larger operator and more training data.
- Shared-encoder speedup is an architectural result available to policy-first methods as well.
- The market-clearing speedup becomes economically important only when every residual evaluation invokes expensive household and firm solvers.
- Dynamic expectation formation and distribution-transition consistency are not tested here.
- Timings are CPU-specific; rerun the notebook on the intended hardware before using the numerical factors in a paper.


## 7. Takeaways and next experiment

The experiment does identify a potentially significant bilateral-model acceleration, but the contribution should be framed narrowly:

> Encode the two populations once, predict a market-clearing price warm start, and retain exact equilibrium discipline through a small number of implicit corrections.

The next decisive test is not another synthetic price network. It is a small dynamic bilateral model with benchmark household and firm solvers. For each price guess, measure the cost of both best responses; then compare cold root solving against shared encoding plus learned warm starts. If the $4\times$ online gain survives while Euler/KKT and clearing residuals remain unchanged, the method has genuine computational value.
